# pressure correction method using SIMPLE algorithm

## Nodes mass flow rates

3. m_a + + m_c - m_b - m_d - m_e = 0  
6. m_d + m_e + m_g + m_i - m_f - m_h = 0  
8. m_h + m_j - m_k = 0

In [9]:
p1 = 500
p2 = 300
p4 = 50
p5 = 350
p7 = 90
p9 = 150

m_k = 70

D_a = 0.5
D_b = 0.3
D_c = 0.2
D_d = 0.4
D_e = 0.3
D_f = 0.5
D_g = 0.35
D_h = 0.8
D_i = 0.6
D_j = 0.4

## Solution Implementation

We solve the system of mass balance equations using the SIMPLE-like pressure correction method.
The mass flow rate is assumed to be $m = D \cdot \Delta p$.
We define residuals $R$ for each node and solve for pressure corrections $p'$ such that the linearized residuals become zero.

### Why $A \cdot P' = R$ works?

You are asking why solving this system minimizes the residual $R$.

The logic is based on **Linearization of the Mass Balance**.

1.  **The Goal**: We want the net mass flow at every node to be zero.
    $$ \sum \dot{m}_{in} - \sum \dot{m}_{out} = 0 $$

2.  **Current State**: With our current guessed pressures ($P^*$), the mass balance is NOT zero. It equals the Residual ($R$).
    $$ \sum \dot{m}^*_{in} - \sum \dot{m}^*_{out} = R $$
    *(Note: In the code, $R$ is defined as Inflow - Outflow)*

3.  **The Correction**: We want to find a pressure correction $P'$ such that the new pressure $P_{new} = P^* + P'$ makes the residual zero.

    The mass flow $\dot{m}$ depends on pressure difference: $\dot{m} = D \cdot (P_{upstream} - P_{downstream})$.
    
    If we change pressures by $P'$, the mass flow changes by:
    $$ \dot{m}_{new} = \dot{m}^* + \dot{m}' $$
    $$ \dot{m}' = D \cdot (P'_{upstream} - P'_{downstream}) $$

4.  **Substituting into Mass Balance**:
    We want the new mass balance to be zero:
    $$ (\sum \dot{m}^*_{in} + \sum \dot{m}'_{in}) - (\sum \dot{m}^*_{out} + \sum \dot{m}'_{out}) = 0 $$
    
    Rearranging terms:
    $$ \underbrace{(\sum \dot{m}^*_{in} - \sum \dot{m}^*_{out})}_{R} + \underbrace{(\sum \dot{m}'_{in} - \sum \dot{m}'_{out})}_{\text{Change due to } P'} = 0 $$
    
    $$ R + (\text{Terms involving } P') = 0 $$
    
    This leads to:
    $$ -(\text{Terms involving } P') = R $$

    This is exactly the system $A \cdot P' = R$.
    
    *   **$R$** is the "excess mass" we need to remove.
    *   **$A \cdot P'$** represents how much mass moves out/in when we change pressures.
    *   By solving this, we find the exact $P'$ needed to cancel out $R$.

In [14]:
import numpy as np

# Initialize unknown pressures
p3 = 0.0
p6 = 0.0
p8 = 0.0

# Relaxation factor (1.0 for linear systems)
alpha = 1.0

print("Starting SIMPLE algorithm iterations...")
print("-" * 50)

for i in range(10):
    # 1. Calculate mass flows based on current pressures
    # Flow = D * (P_upstream - P_downstream)

    # Node 3 connections
    # IN
    m_a = D_a * (p1 - p3)
    m_c = D_c * (p4 - p3)
    # OUT
    m_b = D_b * (p3 - p2)
    m_d = D_d * (p3 - p6)
    m_e = D_e * (p3 - p6)

    # Node 6 connections
    # IN
    # m_d, m_e calculated above
    m_g = D_g * (p7 - p6)
    m_i = D_i * (p9 - p6)
    # OUT
    m_f = D_f * (p6 - p5)
    m_h = D_h * (p6 - p8)

    # Node 8 connections
    # IN
    # m_h calculated above
    m_j = D_j * (p9 - p8)
    # OUT
    # m_k is fixed

    # 2. Calculate Residuals (Net Mass Flow Imbalance)
    # R = Inflow - Outflow
    R3 = (m_a + m_c) - (m_b + m_d + m_e)
    R6 = (m_d + m_e + m_g + m_i) - (m_f + m_h)
    R8 = (m_h + m_j) - m_k

    print(f"Iteration {i + 1}:")
    print(f"  Current Pressures: p3={p3:.2f}, p6={p6:.2f}, p8={p8:.2f}")
    print(f"  Residuals: R3={R3:.4f}, R6={R6:.4f}, R8={R8:.4f}")

    # Check convergence
    if max(abs(R3), abs(R6), abs(R8)) < 1e-9:
        print("  Converged!")
        break

    # 3. Build Pressure Correction System A * P' = R
    # Coefficients derived from mass balance equations

    # Node 3: P'3 * (sum D_out + sum D_in) - P'6 * (D_d + D_e) = R3
    a33 = (D_a + D_c) + (D_b + D_d + D_e)
    a36 = -(D_d + D_e)
    a38 = 0

    # Node 6: -P'3 * (D_d + D_e) + P'6 * (sum D) - P'8 * D_h = R6
    a63 = -(D_d + D_e)
    a66 = (D_d + D_e + D_g + D_i) + (D_f + D_h)
    a68 = -D_h

    # Node 8: -P'6 * D_h + P'8 * (sum D) = R8
    a83 = 0
    a86 = -D_h
    a88 = D_h + D_j  # m_k is fixed, so no derivative

    A = np.array([[a33, a36, a38], [a63, a66, a68], [a83, a86, a88]])
    B = np.array([R3, R6, R8])

    # Solve for corrections
    p_prime = np.linalg.solve(A, B)

    print(
        f"  Corrections: p'3={p_prime[0]:.4f}, p'6={p_prime[1]:.4f}, p'8={p_prime[2]:.4f}"
    )

    # 4. Correct Pressures
    p3 += alpha * p_prime[0]
    p6 += alpha * p_prime[1]
    p8 += alpha * p_prime[2]
    print("-" * 50)

print("\nFinal Results:")
print(f"p3 = {p3:.2f}")
print(f"p6 = {p6:.2f}")
print(f"p8 = {p8:.2f}")

print("\nFinal Mass Flows:")
print(f"m_a (1->3) = {D_a * (p1 - p3):.2f}")
print(f"m_c (2->3) = {D_c * (p2 - p3):.2f}")
print(f"m_b (3->4) = {D_b * (p3 - p4):.2f}")
print(f"m_d (3->6) = {D_d * (p3 - p6):.2f}")
print(f"m_e (3->6) = {D_e * (p3 - p6):.2f}")
print(f"m_g (5->6) = {D_g * (p5 - p6):.2f}")
print(f"m_i (7->6) = {D_i * (p7 - p6):.2f}")
print(f"m_f (6->0) = {D_f * (p6 - 0):.2f}")
print(f"m_h (6->8) = {D_h * (p6 - p8):.2f}")
print(f"m_j (9->8) = {D_j * (p9 - p8):.2f}")
print(f"m_k (8->out) = {m_k:.2f}")

Starting SIMPLE algorithm iterations...
--------------------------------------------------
Iteration 1:
  Current Pressures: p3=0.00, p6=0.00, p8=0.00
  Residuals: R3=350.0000, R6=296.5000, R8=-10.0000
  Corrections: p'3=289.8342, p'6=203.8830, p'8=127.5887
--------------------------------------------------
Iteration 2:
  Current Pressures: p3=289.83, p6=203.88, p8=127.59
  Residuals: R3=0.0000, R6=0.0000, R8=0.0000
  Converged!

Final Results:
p3 = 289.83
p6 = 203.88
p8 = 127.59

Final Mass Flows:
m_a (1->3) = 105.08
m_c (2->3) = 2.03
m_b (3->4) = 71.95
m_d (3->6) = 34.38
m_e (3->6) = 25.79
m_g (5->6) = 51.14
m_i (7->6) = -68.33
m_f (6->0) = 101.94
m_h (6->8) = 61.04
m_j (9->8) = 8.96
m_k (8->out) = 70.00
